# Crop Disease Diagnosis — CNN Image Classification (MobileNetV2)

**Dataset:** [New Plant Diseases Dataset](https://www.kaggle.com/datasets/vipoooool/new-plant-diseases-dataset) — ~87,000 RGB leaf images across 38 classes (14 crop species, healthy + various diseases).

### About this notebook
Same structure and safeguards as the prescription-recognition notebook this project is based on: a single shared `CLASS_LIST`, a stratified split so every class appears in train/val/test, class weighting for imbalance, a mode-collapse diagnostic, TTA, and an interactive upload predictor. This is **Model A** of a two-model system — it covers agricultural field crops only. Model B (indoor/home plants) is a separate notebook.

### Structure
1. Download the dataset
2. Import libraries
3. Data loading & preprocessing, including the stratified split
4. Image generators
5. Transfer learning — MobileNetV2 (two-phase)
6. Evaluation — test accuracy, classification report, confusion matrix, diagnostics
7. Try it — predict on a new leaf photo

## 1. Download the Dataset

Requires a `kaggle.json` API token uploaded to the Colab session (Kaggle → Settings → Create New API Token).

In [ ]:
!pip install -q kaggle
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

!kaggle datasets download -d vipoooool/new-plant-diseases-dataset


In [ ]:
!unzip -q -o new-plant-diseases-dataset.zip -d ./crop_data


### Explore the folder structure

Run this first and check the printed output before adjusting the paths in Section 3 below.

In [ ]:
import os

for root, dirs, files in os.walk("./crop_data"):
    level = root.replace("./crop_data", "").count(os.sep)
    indent = "  " * level
    if level <= 3:
        print(f"{indent}{os.path.basename(root)}/")
        if len(files) > 0 and level == 3:
            print(f"{indent}  ... ({len(files)} files total)")


## 2. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mobilenet_preprocess

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix


## 3. Data Loading & Preprocessing

This dataset ships as folders-of-images (one folder per crop-disease class) rather than a CSV index, so Section 3 walks every subfolder to build the same kind of `filepath` / `label` table the prescription notebook built from its CSV. Everything downstream (the master `CLASS_LIST`, the stratified split, class weighting) works identically once that table exists.

The dataset ships pre-split into `train/` and `valid/` folders, each containing one subfolder per class (e.g. `Tomato___Early_blight`). Both are pooled below into a single table, then re-split ourselves so the test set is guaranteed labeled and stratified (see the split step further down).

In [ ]:
# Walk the extracted dataset and collect every image path + its class-folder label.
# This plays the same role the CSV played in the prescription notebook.
filepaths = []
labels = []

for root, dirs, files in os.walk("./crop_data"):
    for fname in files:
        if fname.lower().endswith(('.png', '.jpg', '.jpeg')):
            filepaths.append(os.path.join(root, fname))
            labels.append(os.path.basename(root))

data = pd.DataFrame({"filepath": filepaths, "label": labels})

missing_values = data.isnull().sum()
print("Missing values per column:\n", missing_values)

print("\nDuplicate rows:", data.duplicated().sum())
data = data.drop_duplicates().reset_index(drop=True)

print("\nTotal images found:", len(data))
print("Number of classes:", data["label"].nunique())
print("\nImages per class (min / median / max):",
      data["label"].value_counts().min(), '/',
      int(data["label"].value_counts().median()), '/',
      data["label"].value_counts().max())


### Output layer size & the master class list

The number of neurons in the output layer = number of target classes, with softmax — not a single sigmoid neuron.

`CLASS_LIST` is defined **once**, sorted, and passed explicitly to every generator (`classes=CLASS_LIST`) — this guarantees a consistent label-to-index mapping between training, validation, and test, exactly like the prescription notebook.

In [ ]:
NUM_CLASSES = data["label"].nunique()
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
CLASS_LIST = sorted(data["label"].unique())

print(f"Output layer size: {NUM_CLASSES} neurons (one per crop-disease class)")


### Stratified train / validation / test split — the critical fix

Just like the prescription notebook, the split is done with `train_test_split(..., stratify=...)` rather than slicing folders by position. This guarantees every class is represented in training, validation, **and** test, in proportion — the fix for the 0-recall-on-rare-classes bug. The dataset's own pre-made `train`/`valid` folders are pooled first and then re-split 80/10/10 so the test set is guaranteed to be labeled and stratified too.

In [ ]:
train_df, test_df = train_test_split(
    data,
    test_size=0.10,
    stratify=data["label"],
    random_state=42
)
train_df, val_df = train_test_split(
    train_df,
    test_size=0.1111,  # 0.1111 of the remaining 90% ≈ 10% of the original total
    stratify=train_df["label"],
    random_state=42
)

print("Training rows:", len(train_df), " | Validation rows:", len(val_df), " | Test rows:", len(test_df))
print("Classes present in training:", train_df["label"].nunique(), "/", NUM_CLASSES)
print("Classes present in validation:", val_df["label"].nunique(), "/", NUM_CLASSES)
print("Classes present in test:", test_df["label"].nunique(), "/", NUM_CLASSES)


### Class weights

Even with a stratified split, some classes have noticeably fewer samples than others. `class_weight` tells the model to pay proportionally more attention to under-represented classes during training, instead of implicitly favoring the most common ones.

In [ ]:
class_weight_values = compute_class_weight(
    class_weight='balanced',
    classes=np.array(CLASS_LIST),
    y=train_df["label"]
)
class_weight_dict = {i: w for i, w in enumerate(class_weight_values)}
print("Class weight range:", round(min(class_weight_values), 2), "to", round(max(class_weight_values), 2))


## 4. Image Generators

MobileNetV2 needs its own `preprocess_input`, which scales pixels to `[-1, 1]` (different from a plain `rescale=1./255`).

Validation and test generators skip augmentation — augmentation belongs only on the training set.

In [ ]:
# --- MobileNetV2 generators (its own preprocess_input, scales to [-1, 1]) ---
train_datagen = ImageDataGenerator(
    preprocessing_function=mobilenet_preprocess,
    rotation_range=20,
    zoom_range=0.15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest',
)
val_test_datagen = ImageDataGenerator(preprocessing_function=mobilenet_preprocess)

train_generator = train_datagen.flow_from_dataframe(
    dataframe=train_df, x_col='filepath', y_col='label',
    classes=CLASS_LIST, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='sparse', shuffle=True
)
val_generator = val_test_datagen.flow_from_dataframe(
    dataframe=val_df, x_col='filepath', y_col='label',
    classes=CLASS_LIST, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='sparse', shuffle=False
)
test_generator = val_test_datagen.flow_from_dataframe(
    dataframe=test_df, x_col='filepath', y_col='label',
    classes=CLASS_LIST, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='sparse', shuffle=False
)

class_names = CLASS_LIST


### Quick sanity check — view a batch

In [ ]:
images, labels_batch = next(train_generator)
idx_to_class = {i: c for i, c in enumerate(CLASS_LIST)}

plt.figure(figsize=(10, 6))
for i in range(8):
    plt.subplot(2, 4, i + 1)
    # Un-normalize from MobileNetV2's [-1, 1] preprocessing back to [0, 1] just for display
    plt.imshow((images[i] + 1) / 2)
    plt.title(idx_to_class[int(labels_batch[i])], fontsize=8)
    plt.axis('off')
plt.tight_layout()
plt.show()


## 5. Transfer Learning — MobileNetV2

MobileNetV2 is a lightweight pretrained architecture, which generalizes well on small-to-medium datasets since there's less capacity to overfit with. The convolutional base is pretrained on ImageNet and frozen; only a small new classification head is trained on top.

In [ ]:
base_model = MobileNetV2(include_top=False, input_shape=(224, 224, 3), weights='imagenet')
base_model.trainable = False

crop_model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(512, activation='relu'),
    Dropout(0.3),
    Dense(NUM_CLASSES, activation='softmax')
])

crop_model.summary()


In [ ]:
crop_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
                     loss='sparse_categorical_crossentropy',
                     metrics=['accuracy'])

early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-7, verbose=1)

history_phase1 = crop_model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=15,
    class_weight=class_weight_dict,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)


### 5.1 Fine-tuning MobileNetV2 (Phase 2)

Unfreeze the top layers of the base model and continue training at a much lower learning rate, so the pretrained weights adapt toward this domain instead of staying generic to ImageNet photos. This continues training the *same* `crop_model`.

In [ ]:
# Unfreeze all but the first 40 layers, which hold the most generic, low-level
# ImageNet features (edges/colors/textures) that transfer regardless of domain.
base_model.trainable = True
for layer in base_model.layers[:40]:
    layer.trainable = False

crop_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
                     loss='sparse_categorical_crossentropy',
                     metrics=['accuracy'])

finetune_early_stop = EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True)
finetune_reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7, verbose=1)

history_phase2 = crop_model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=20,
    class_weight=class_weight_dict,
    callbacks=[finetune_early_stop, finetune_reduce_lr],
    verbose=1
)

crop_model.save("crop_disease_mobilenetv2.keras")


## 6. Evaluation

### 6.1 Training curves

In [ ]:
val_acc = history_phase1.history['val_accuracy'] + history_phase2.history['val_accuracy']
val_loss = history_phase1.history['val_loss'] + history_phase2.history['val_loss']

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))

ax[0].plot(val_acc, label='MobileNetV2', color='#2E7D32')
ax[0].set_title('Validation Accuracy over Epochs')
ax[0].set_xlabel('Epoch'); ax[0].set_ylabel('Accuracy'); ax[0].legend()

ax[1].plot(val_loss, label='MobileNetV2', color='#2E7D32')
ax[1].set_title('Validation Loss over Epochs')
ax[1].set_xlabel('Epoch'); ax[1].set_ylabel('Loss'); ax[1].legend()

plt.tight_layout()
plt.show()


### 6.2 Test set performance

In [ ]:
test_loss, test_acc = crop_model.evaluate(test_generator, verbose=0)
print(f"MobileNetV2 — Test Accuracy: {test_acc:.4f}")


### 6.3 Classification report & confusion matrix

Using **test-time augmentation (TTA)**: each test image is predicted several times with light random augmentation applied, and the probabilities are averaged. This tends to smooth out noisy per-image predictions and lift recall on the weaker classes, at the cost of a few extra minutes of inference.

In [ ]:
TTA_ROUNDS = 5

tta_datagen = ImageDataGenerator(
    preprocessing_function=mobilenet_preprocess,
    rotation_range=8, zoom_range=0.08,
    width_shift_range=0.05, height_shift_range=0.05
)

y_test = test_generator.classes
tta_probs = np.zeros((len(y_test), NUM_CLASSES))

for round_i in range(TTA_ROUNDS):
    tta_generator = tta_datagen.flow_from_dataframe(
        dataframe=test_df, x_col='filepath', y_col='label',
        classes=CLASS_LIST, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
        class_mode='sparse', shuffle=False
    )
    tta_probs += crop_model.predict(tta_generator, verbose=0)
    print(f"TTA round {round_i + 1}/{TTA_ROUNDS} done")

y_pred_prob = tta_probs / TTA_ROUNDS
y_pred = np.argmax(y_pred_prob, axis=1)

print(classification_report(y_test, y_pred, target_names=class_names))


### 6.4 Mode-collapse diagnostic

If recall is 0 for most classes, check the output below: if the model is predicting only a handful of classes for *every* test image, that's mode collapse (a training-stability problem), not a labeling bug.

In [ ]:
unique_preds, counts = np.unique(y_pred, return_counts=True)
print(f"Number of distinct classes actually predicted: {len(unique_preds)} / {NUM_CLASSES}")
top10 = sorted(zip(unique_preds, counts), key=lambda x: -x[1])[:10]
print("Most frequently predicted classes:")
for idx, cnt in top10:
    print(f"  {class_names[idx]}: predicted {cnt} times")


In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(14, 12))
sns.heatmap(cm, annot=False, cmap="YlGn", xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted'); plt.ylabel('Actual')
plt.title('Confusion Matrix — Crop Leaf')
plt.xticks(rotation=90, fontsize=6); plt.yticks(rotation=0, fontsize=6)
plt.tight_layout()
plt.show()


## 7. Try It — Predict on a New Crop Leaf Photo

Upload a leaf photo and get the model's top-3 guesses.

In [ ]:
from google.colab import files
from tensorflow.keras.preprocessing import image

def predict_crop_leaf(img_path, top_k=3, tta_rounds=5):
    img = image.load_img(img_path, target_size=IMG_SIZE)
    img_array = image.img_to_array(img)

    # Test-time augmentation: average predictions over a few light random transforms,
    # plus the original image, for a steadier estimate on a single sample.
    tta_gen = ImageDataGenerator(rotation_range=8, zoom_range=0.08,
                                  width_shift_range=0.05, height_shift_range=0.05)
    batch = np.stack([img_array] + [tta_gen.random_transform(img_array) for _ in range(tta_rounds - 1)])
    batch = mobilenet_preprocess(batch)  # match training preprocessing ([-1, 1] scaling)

    probs = crop_model.predict(batch).mean(axis=0)
    top_indices = probs.argsort()[-top_k:][::-1]

    plt.imshow(img)
    plt.axis('off')
    plt.title(f"Top guess: {class_names[top_indices[0]]} ({probs[top_indices[0]]:.1%})")
    plt.show()

    print("Top predictions:")
    for i in top_indices:
        print(f"  {class_names[i]}: {probs[i]:.1%}")

uploaded = files.upload()
for filename in uploaded.keys():
    predict_crop_leaf(filename)
